In [9]:
# Mount Google Drive (where the VIDVRD zip + outputs live)
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Clone repo (edit URL if you use a fork)
%cd /content
!rm -rf STTran
!git clone <YOUR_GITHUB_REPO_URL> STTran
%cd /content/STTran


/content
Cloning into 'STTran'...
remote: Enumerating objects: 799, done.
remote: Counting objects: 100% (799/799), done.
remote: Compressing objects: 100% (543/543), done.
remote: Total 799 (delta 293), reused 732 (delta 226), pack-reused 0 (from 0)
Receiving objects: 100% (799/799), 10.40 MiB | 20.81 MiB/s, done.
Resolving deltas: 100% (293/293), done.
/content/STTran


In [11]:
%%bash
set -euo pipefail
set -x

cd /content/STTran
# Idempotent: reruns should be fast (uses .colab_setup_stamp.json)
python -u setup_colab.py --colab


[colab] Skipping legacy native extension build (use --compile-native or STTRAN_COMPILE_NATIVE=1 to force).
STTran Colab setup  repo_root=/content/STTran
+ /usr/bin/python3 -m pip install --upgrade pip
[colab] keeping existing torch 2.10.0+cu128 (cuda=True)
+ /usr/bin/python3 -m pip install numpy scipy imageio pillow tqdm six cython ninja matplotlib networkx gdown pyyaml opencv-python pandas dill easydict h5py
[download] http://nlp.stanford.edu/data/glove.6B.zip -> /content/STTran/data/glove.6B.zip
+ /usr/bin/python3 /content/STTran/scripts/download_sttran_ag_weights.py --out_dir /content/STTran/.sttran_weight_cache --link_into_repo
[download] /content/STTran/.sttran_weight_cache/faster_rcnn_ag.pth
[download] /content/STTran/.sttran_weight_cache/object_bbox_and_relationship_filtersmall.pkl
[download] /content/STTran/.sttran_weight_cache/sttran_predcls.tar
[download] /content/STTran/.sttran_weight_cache/sttran_sgcls.tar
[download] /content/STTran/.sttran_weight_cache/sttran_sgdet.tar

Do

+ cd /content/STTran
+ python setup_colab.py --colab
Downloading...
From (original): https://drive.google.com/uc?id=1-u930Pk0JYz3ivS6V_HNTM1D5AxmN5Bs
From (redirected): https://drive.google.com/uc?id=1-u930Pk0JYz3ivS6V_HNTM1D5AxmN5Bs&confirm=t&uuid=04e0caa3-6d51-44f8-b3b7-377b39eca442
To: /content/STTran/.sttran_weight_cache/faster_rcnn_ag.pth
100%|██████████| 380M/380M [00:13<00:00, 28.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=19BkAwjCw5ByyGyZjFo174Oc3Ud56fkaT
From (redirected): https://drive.google.com/uc?id=19BkAwjCw5ByyGyZjFo174Oc3Ud56fkaT&confirm=t&uuid=9ad427a6-acc8-465a-9bc4-ec48c7c6160f
To: /content/STTran/.sttran_weight_cache/object_bbox_and_relationship_filtersmall.pkl
100%|██████████| 136M/136M [00:03<00:00, 44.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Sk5qFLWTZmwr63fHpy_C7oIxZSQU16vU
From (redirected): https://drive.google.com/uc?id=1Sk5qFLWTZmwr63fHpy_C7oIxZSQU16vU&confirm=t&uuid=dd1a45f8-f541-446a-8021-b9554924d095


In [12]:
# Optional: if you keep big weights on Drive instead of inside the repo, set these.
# Otherwise, skip this cell.
import os

# os.environ["FASTER_RCNN_AG_PTH"] = "/content/drive/MyDrive/weights/faster_rcnn_ag.pth"
# os.environ["STTRAN_CKPT"] = "/content/drive/MyDrive/weights/sttran_predcls.tar"


In [13]:
%%bash
set -euo pipefail
set -x

# Unzip VIDVRD once (fast local SSD), then copy to Drive (one-time cost).
# This avoids Drive-unzip hangs and avoids re-unzipping every run.
ZIP="/content/drive/MyDrive/vidvrd-dataset_480.zip"
DRIVE_ROOT="/content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480"

# If already present on Drive, do nothing.
if [ -d "$DRIVE_ROOT/test_frames_480" ] && [ -d "$DRIVE_ROOT/test_480" ] && \
   [ -d "$DRIVE_ROOT/train_frames_480" ] && [ -d "$DRIVE_ROOT/train_480" ]; then
  echo "[skip] dataset already present on Drive: $DRIVE_ROOT"
  exit 0
fi

# Clean local extract targets
rm -rf /content/VIDVRD-DATASET_480 /content/VIDVRD /content/VIDVRD-DATASET_480

# Unzip to local (/content)
unzip -q "$ZIP" -d /content

# Auto-detect extracted root (handles common nesting variants)
LOCAL_ROOT=""
for CAND in \
  /content/VIDVRD-DATASET_480 \
  /content/VIDVRD-DATASET_480/VIDVRD-DATASET_480 \
  /content/VIDVRD/VIDVRD-DATASET_480 \
  /content/VIDVRD-DATASET_480/vidvrd-dataset_480 \
  /content/vidvrd-dataset_480 \
  /content/VIDVRD-DATASET_480_480 \
  ; do
  if [ -d "$CAND" ]; then
    LOCAL_ROOT="$CAND"
    break
  fi
done

if [ -z "$LOCAL_ROOT" ]; then
  echo "[debug] /content listing:" >&2
  ls -lah /content | head -n 200 >&2
  echo "[error] Could not find extracted VIDVRD root under /content. Check ZIP structure." >&2
  exit 2
fi

echo "[local] dataset root: $LOCAL_ROOT"

# Copy to Drive with progress
mkdir -p "$(dirname "$DRIVE_ROOT")"
rsync -a --info=progress2 "$LOCAL_ROOT/" "$DRIVE_ROOT/"

echo "[ok] dataset_root_drive=$DRIVE_ROOT"


[unzip] /content/drive/MyDrive/vidvrd-dataset_480.zip -> /content/drive/MyDrive/VIDVRD/_vidvrd_extract_tmp


KeyboardInterrupt: 

In [17]:
# Quick sanity check: confirm dataset exists on Drive
from pathlib import Path

DATASET_ROOT = Path("/content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480")
print("DATASET_ROOT:", DATASET_ROOT)
print("has test_frames_480:", (DATASET_ROOT / "test_frames_480").is_dir())
print("has test_480:", (DATASET_ROOT / "test_480").is_dir())
print("has train_frames_480:", (DATASET_ROOT / "train_frames_480").is_dir())
print("has train_480:", (DATASET_ROOT / "train_480").is_dir())


OUT MB: 0
TMP MB: 64027

Watching size for ~60s (should increase if unzip is working)...
t= 0s  OUT=0MB  TMP=64027MB
t=10s  OUT=0MB  TMP=64027MB
t=20s  OUT=0MB  TMP=64027MB
t=30s  OUT=0MB  TMP=64027MB
t=40s  OUT=0MB  TMP=64027MB
t=50s  OUT=0MB  TMP=64027MB


In [ ]:
%%bash
set -euo pipefail
set -x

cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480" \
  --video_id "ILSVRC2015_train_00010001" \
  --expected_hw "480,854" \
  --max_frames 32 \
  --no_forward \
  --num_predicates 0


[remove tmp] /content/drive/MyDrive/VIDVRD/_vidvrd_extract_tmp


In [ ]:
%%bash
set -euo pipefail
set -x

# Optional: run a forward pass too (requires weights present).
cd /content/STTran
python -u lib/vidvrd_pipeline_validate.py \
  --dataset_root "/content/drive/MyDrive/VIDVRD/VIDVRD-DATASET_480" \
  --video_id "ILSVRC2015_train_00010001" \
  --expected_hw "480,854" \
  --max_frames 32 \
  --num_predicates 0


In [16]:
%cd /content/STTran
!python colab/vidvrd_train_colab.py \
  --out_dir "/content/drive/MyDrive/vidvrd_runs/smoke_synth" \
  --epochs 1 \
  --synthetic


Process is interrupted.


In [ ]:
# (Optional) Quick check: print GPU availability
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


In [ ]:
# NOTE: do NOT use --dataset_zip in normal runs.
# After you have the dataset on Drive, always run the validator with --dataset_root.
pass


In [ ]:
# Placeholder cell (kept to minimize diffs). You can ignore it.
pass


In [ ]:
# Placeholder cell (kept to minimize diffs). You can ignore it.
pass
